# 🧠 Text2SQL Fine-tuning with GRPO: Improving SQL Schema Fidelity in Small Language Models

> **Dataset:** [Spider](https://yale-seas.yale.edu/spider/) | [BIRD](https://bird-bench.github.io/)  
> **Framework:** 🤗 TRL + PEFT + Unsloth  
> **Tags:** `text2sql` `grpo` `rlhf` `fine-tuning` `slm` `sql` `nlp`

---

## 📌 Problem Statement

Natural language to SQL (Text2SQL) is critical for making databases accessible to non-technical users. While Large Language Models (LLMs) perform well, they are expensive. 

**Small Language Models (SLMs)** are faster and cheaper but suffer from:

- 🔴 **Hallucinating** table/column names not in the schema
- 🔴 **Poor schema alignment** — referencing wrong or non-existent fields
- 🔴 **Inconsistent formatting** — not returning valid SQL code blocks

This notebook fine-tunes an SLM using **Group Relative Policy Optimization (GRPO)** with a custom multi-component reward function to improve SQL schema fidelity and compare it against a zero-shot baseline.

---

## 🎯 Objectives & Key Results (OKRs)

### Objective 1 — Improve SQL Schema Fidelity
> Generate SQL that correctly references **only** tables and columns defined in the schema.

| Key Result | Metric | Baseline | Target |
|---|---|---|---|
| KR 1.1 | Schema fidelity score (avg) | ~0.45 | ≥ 0.75 |
| KR 1.2 | % completions with 100% valid table refs | ~40% | ≥ 70% |
| KR 1.3 | % completions with 100% valid column refs | ~35% | ≥ 65% |

### Objective 2 — Improve SQL Format Correctness
> Ensure the model always returns parseable SQL inside a \`\`\`sql code block.

| Key Result | Metric | Baseline | Target |
|---|---|---|---|
| KR 2.1 | Format reward (avg) | ~0.55 | ≥ 0.90 |
| KR 2.2 | % responses with valid SQL fence | ~55% | ≥ 90% |
| KR 2.3 | % responses with parseable SQL | ~50% | ≥ 85% |

### Objective 3 — Improve SQL Executability
> Generated SQL should execute without runtime errors.

| Key Result | Metric | Baseline | Target |
|---|---|---|---|
| KR 3.1 | Exec reward (avg) | ~0.40 | ≥ 0.70 |
| KR 3.2 | % queries that execute without error | ~40% | ≥ 70% |

### Objective 4 — Validate Across Benchmarks
> Generalise improvements across Spider and BIRD datasets.

| Key Result | Metric | Baseline | Target |
|---|---|---|---|
| KR 4.1 | Combined reward on Spider dev set | ~0.45 | ≥ 0.72 |
| KR 4.2 | Combined reward on BIRD dev set | ~0.40 | ≥ 0.68 |
| KR 4.3 | Performance gap Spider → BIRD | > 20% | < 10% |

---

## ⚙️ Reward Function Design

The GRPO training signal is a weighted combination of three reward components:

```
combined_reward = (
    0.20 × format_reward         # Is the SQL in a valid code block and parseable?
  + 0.50 × exec_reward           # Does the SQL execute without errors?
  + 0.30 × schema_fidelity       # Does SQL only reference valid schema tables/columns?
)
```

| Reward | Score | Meaning |
|---|---|---|
| `format` | `1.0` | Valid ` ```sql ``` ` fence + parseable by sqlglot |
| `format` | `0.0` | No SQL or parse error |
| `exec` | `1.0` | Executes successfully on SQLite |
| `exec` | `0.0` | SQL execution error |
| `exec` | `-1.0` | No SQL found |
| `schema` | `1.0` | All referenced tables/columns exist in schema |
| `schema` | `0.5` | No schema provided (neutral) |
| `schema` | `0.0` | No SQL or all references invalid |

---

## 🗂️ Notebook Structure

| Cell | Section | Description |
|---|---|---|
| 1 | ⚙️ Setup | Install dependencies |
| 2 | 📥 Data Download | Download Spider + BIRD datasets |
| 3 | 🔍 Data Exploration | Inspect samples, schema distributions |
| 4 | 🏗️ Schema Serialization | Convert schemas to prompt format |
| 5 | 🎯 Reward Functions | Define + test format / exec / schema rewards |
| 6 | 🏋️ GRPO Training | Fine-tune SLM with reward-guided RL |
| 7 | 📊 Evaluation | Compare baseline vs fine-tuned on dev sets |
| 8 | 📈 Results | Visualize OKR metrics and score distributions |

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install loguru sqlglot sqlparse

---

## 📥 Section 2 — Data Download

Download and extract the Spider and BIRD benchmark datasets. Both are unzipped into `/kaggle/working/rawdata/` for consistent path access throughout the notebook.

In [2]:
!rm -rf /kaggle/working/rawdata

In [3]:
from __future__ import annotations

import argparse
import os
import shutil
import zipfile
from pathlib import Path
import sqlite3
import requests
from loguru import logger
from tqdm import tqdm

SPIDER_URL = "https://drive.usercontent.google.com/download?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J&export=download&confirm=t"
BIRD_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"

DATASETS = {
    "spider": SPIDER_URL,
    "bird": BIRD_URL,
}

def _download(url: str, dest: Path) -> None:
    """Stream download *url* to *dest*."""
    logger.info(f"Downloading {url} → {dest}")
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    with open(dest, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            fh.write(chunk)
            bar.update(len(chunk))

def _extract(archive: Path, dest: Path) -> None:
    logger.info(f"Extracting {archive} → {dest}")
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(dest)

def download_datasets(output_dir: str) -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    tmp = out / "_downloads"
    tmp.mkdir(exist_ok=True)

    for name, url in DATASETS.items():
        archive = tmp / f"{name}.zip"
        extract_dir = out / name
        if extract_dir.exists():
            logger.info(f"{name} already extracted at {extract_dir}, skipping.")
            continue
        _download(url, archive)
        _extract(archive, extract_dir)
        archive.unlink(missing_ok=True)

    shutil.rmtree(tmp, ignore_errors=True)
    logger.success(f"All datasets ready in {out}")


output_dir = "/kaggle/working/"
download_datasets(output_dir + "rawdata")

2026-03-09 12:47:19.314 | INFO     | __main__:_download:23 - Downloading https://drive.usercontent.google.com/download?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J&export=download&confirm=t → /kaggle/working/rawdata/_downloads/spider.zip
100%|██████████| 206M/206M [00:03<00:00, 62.2MB/s] 
2026-03-09 12:47:23.517 | INFO     | __main__:_extract:33 - Extracting /kaggle/working/rawdata/_downloads/spider.zip → /kaggle/working/rawdata/spider
2026-03-09 12:47:30.503 | INFO     | __main__:_download:23 - Downloading https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip → /kaggle/working/rawdata/_downloads/bird.zip
100%|██████████| 346M/346M [00:58<00:00, 5.89MB/s]   
2026-03-09 12:48:31.008 | INFO     | __main__:_extract:33 - Extracting /kaggle/working/rawdata/_downloads/bird.zip → /kaggle/working/rawdata/bird
2026-03-09 12:48:31.738 | SUCCESS  | __main__:download_datasets:54 - All datasets ready in /kaggle/working/rawdata


In [4]:
!unzip -q /kaggle/working/rawdata/bird/dev_20240627/dev_databases.zip -d /kaggle/working/rawdata/bird/

In [5]:
import json 

def _schema_from_sqlite(db_path: Path) -> dict[str, list[str]]:
    """Extract table→columns mapping from a SQLite file."""
    schema: dict[str, list[str]] = {}
    conn = sqlite3.connect(str(db_path))
    try:
        cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = [row[0] for row in cursor.fetchall()]
        for table in tables:
            col_cursor = conn.execute(f"PRAGMA table_info(`{table}`)")
            schema[table.lower()] = [row[1].lower() for row in col_cursor.fetchall()]
    finally:
        conn.close()
    return schema


def _schema_from_tables_json(tables_json_path: Path) -> dict[str, dict[str, list[str]]]:
    """Parse Spider-style ``tables.json`` into a dict of db_id → schema."""
    with open(tables_json_path) as fh:
        tables_data = json.load(fh)

    result: dict[str, dict[str, list[str]]] = {}
    for db in tables_data:
        db_id = db["db_id"]
        col_names = db.get("column_names_original", db.get("column_names", []))
        table_names = db.get("table_names_original", db.get("table_names", []))

        schema: dict[str, list[str]] = {t.lower(): [] for t in table_names}
        for table_idx, col_name in col_names:
            if table_idx < 0:
                continue  # wildcard column
            tname = table_names[table_idx].lower()
            schema[tname].append(col_name.lower())
        result[db_id] = schema
    return result


def _is_probably_sqlite_file(p: Path) -> bool:
    """Quick check to skip obvious non-database files."""
    # Skip macOS resource-fork files and __MACOSX folder artifacts
    if p.name.startswith("._") or "__MACOSX" in p.parts:
        return False
    if not p.is_file():
        return False
    # Optional header check: SQLite files usually start with this magic string
    try:
        with p.open("rb") as fh:
            header = fh.read(16)
        return header.startswith(b"SQLite format 3")
    except OSError:
        return False

def serialize_schemas(input_dir: str, output_dir: str) -> None:
    in_path = Path(input_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    all_schemas: dict[str, dict[str, list[str]]] = {}

    # Spider / BIRD style: tables.json at root of dataset
    logger.info(f"Parsing Spider/BIRD-style tables.json files under {in_path}...")
    for tables_json in in_path.rglob("dev_tables.json"):
        logger.debug(f"Processing {tables_json}")
        schemas = _schema_from_tables_json(tables_json)
        all_schemas.update(schemas)
    logger.info(f"Number of schemas in BIRD {len(all_schemas)}")
    
    
    # Also scan individual SQLite .db files from SPIDER
    sqlite_exts = {".db", ".sqlite", ".sqlite3", ".sqlllite"}  
    db_files = [p for p in in_path.rglob("*") if _is_probably_sqlite_file(p) and (p.suffix.lower() in sqlite_exts)]
    logger.info(f"Parsing {len(db_files)} SQLite files for schemas...")
    counter = 0
    for db_file in tqdm(db_files, desc="SQLite schemas"):
        db_id = db_file.stem
        if db_id not in all_schemas:
            try:
                all_schemas[db_id] = _schema_from_sqlite(db_file)
                counter+=1
            except Exception as exc:  # noqa: BLE001
                logger.warning(f"Could not read {db_file}: {exc}")
    logger.info(f"Number of schemas in SPIDER {counter}")

    out_file = out_path / "schema_lookup.json"
     # Write as list instead of object
    payload = [
        {"db_id": db_id, "schema": schema}
        for db_id, schema in sorted(all_schemas.items())
    ]
    with open(out_file, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, ensure_ascii=False)
    logger.success(f"Serialised {len(all_schemas)} schemas → {out_file}")

input_dir = "/kaggle/working/rawdata"
output_dir = "/kaggle/working/serialized_data"
serialize_schemas(input_dir, output_dir)

2026-03-09 12:48:42.241 | INFO     | __main__:serialize_schemas:62 - Parsing Spider/BIRD-style tables.json files under /kaggle/working/rawdata...
2026-03-09 12:48:42.273 | DEBUG    | __main__:serialize_schemas:64 - Processing /kaggle/working/rawdata/bird/dev_20240627/dev_tables.json
2026-03-09 12:48:42.277 | INFO     | __main__:serialize_schemas:67 - Number of schemas in BIRD 11
2026-03-09 12:48:42.344 | INFO     | __main__:serialize_schemas:73 - Parsing 383 SQLite files for schemas...
SQLite schemas: 100%|██████████| 383/383 [00:00<00:00, 4917.06it/s]
2026-03-09 12:48:42.424 | INFO     | __main__:serialize_schemas:83 - Number of schemas in SPIDER 205
2026-03-09 12:48:42.431 | SUCCESS  | __main__:serialize_schemas:93 - Serialised 216 schemas → /kaggle/working/serialized_data/schema_lookup.json


---

## 🔍 Section 3 — Data Exploration

Inspect sample counts, schema distributions, SQL complexity, and question types across Spider and BIRD splits.

In [6]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
f =  "/kaggle/working/serialized_data/schema_lookup.json"
with open(f) as fh:
    schemas_ds = pd.read_json(fh)
print(f"Number of schemas: {len(schemas_ds)}")
schemas_ds.head()

Number of schemas: 216


,db_id,schema
0,aan_1,"{'affiliation': ['affiliation_id', 'name', 'address'], 'author': ['author_id', 'name', 'email'], 'author_list': ['paper_id', 'author_id', 'affiliation_id'], 'citation': ['paper_id', 'cited_paper_id'], 'paper': ['paper_id', 'title', 'venue', 'year']}"
1,academic,"{'author': ['aid', 'homepage', 'name', 'oid'], 'conference': ['cid', 'homepage', 'name'], 'domain': ['did', 'name'], 'domain_author': ['aid', 'did'], 'domain_conference': ['cid', 'did'], 'journal': ['homepage', 'jid', 'name'], 'domain_journal': ['did', 'jid'], 'keyword': ['keyword', 'kid'], 'domain_keyword': ['did', 'kid'], 'publication': ['abstract', 'cid', 'citation_num', 'jid', 'pid', 'reference_num', 'title', 'year'], 'domain_publication': ['did', 'pid'], 'organization': ['continent', 'homepage', 'name', 'oid'], 'publication_keyword': ['pid', 'kid'], 'writes': ['aid', 'pid'], 'cite': ['cited', 'citing']}"
2,activity_1,"{'activity': ['actid', 'activity_name'], 'participates_in': ['stuid', 'actid'], 'faculty_participates_in': ['facid', 'actid'], 'student': ['stuid', 'lname', 'fname', 'age', 'sex', 'major', 'advisor', 'city_code'], 'faculty': ['facid', 'lname', 'fname', 'rank', 'sex', 'phone', 'room', 'building']}"
3,address_1,"{'student': ['stuid', 'lname', 'fname', 'age', 'sex', 'major', 'advisor', 'city_code'], 'direct_distance': ['city1_code', 'city2_code', 'distance'], 'city': ['city_code', 'city_name', 'state', 'country', 'latitude', 'longitude']}"
4,advertising_agencies,"{'agencies': ['agency_id', 'agency_details'], 'staff': ['staff_id', 'agency_id', 'staff_details'], 'clients': ['client_id', 'agency_id', 'sic_code', 'client_details'], 'invoices': ['invoice_id', 'client_id', 'invoice_status', 'invoice_details'], 'meetings': ['meeting_id', 'client_id', 'meeting_outcome', 'meeting_type', 'billable_yn', 'start_date_time', 'end_date_time', 'purpose_of_meeting', 'other_details'], 'payments': ['payment_id', 'invoice_id', 'payment_details'], 'staff_in_meetings': ['meeting_id', 'staff_id']}"


In [7]:
bird_examples_json1 = "/kaggle/working/rawdata/bird/dev_20240627/dev.json"
bird_examples_json2 = "/kaggle/working/rawdata/bird/dev_20240627/dev_tied_append.json"
bird_examples_ds1 = pd.read_json(bird_examples_json1)
bird_examples_ds2 = pd.read_json(bird_examples_json2)
bird_examples_ds = pd.concat([bird_examples_ds1, bird_examples_ds2], axis=0, ignore_index=True)
bird_examples_ds["source"]  = "bird"
bird_examples_ds.drop(columns=["difficulty", "evidence", "question_id"], inplace=True)
print(f"Length of samples: {bird_examples_ds.count()}")
bird_examples_ds.head(2)

Length of samples: db_id       1576
question    1576
SQL         1576
source      1576
dtype: int64


,db_id,question,SQL,source
0,california_schools,What is the highest eligible free rate for K-12 students in the schools in Alameda County?,SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1,bird
1,california_schools,Please list the lowest three eligible free rates for students aged 5-17 in continuation schools.,SELECT `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)` FROM frpm WHERE `Educational Option Type` = 'Continuation School' AND `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)` IS NOT NULL ORDER BY `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)` ASC LIMIT 3,bird


In [8]:
spider_examples_json = "/kaggle/working/rawdata/spider/spider_data/dev.json"
spider_examples_ds = pd.read_json(spider_examples_json)
spider_examples_ds["source"]  = "spider"
spider_examples_ds.drop(columns=["query_toks_no_value", "query_toks", "question_toks", "sql"], inplace=True)
spider_examples_ds.rename(columns={"query": "SQL"}, inplace=True)
spider_examples_ds.head(3)

,db_id,SQL,question,source
0,concert_singer,SELECT count(*) FROM singer,How many singers do we have?,spider
1,concert_singer,SELECT count(*) FROM singer,What is the total number of singers?,spider
2,concert_singer,"SELECT name , country , age FROM singer ORDER BY age DESC","Show name, country, age for all singers ordered by age from the oldest to the youngest.",spider


In [9]:
examples_ds = pd.concat([spider_examples_ds, bird_examples_ds], axis=0, ignore_index=True)
print(f"Total Examples: {len(examples_ds)}")
print("="*20)
print(f"Examples by Source: {examples_ds["source"].value_counts()}")

Total Examples: 2610
Examples by Source: source
bird      1576
spider    1034
Name: count, dtype: int64


---

## 🏗️ Section 4 — Schema Serialization

Convert raw Spider/BIRD table schemas into structured prompt-ready dictionaries mapping table names → column lists.

In [10]:
# Merge based on db_id
merged_samples = pd.merge(schemas_ds, examples_ds, on="db_id", how="inner")
merged_samples.head(3)

,db_id,schema,SQL,question,source
0,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}",SELECT count(*) FROM ship WHERE disposition_of_ship = 'Captured',How many ships ended up being 'Captured'?,spider
1,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}","SELECT name , tonnage FROM ship ORDER BY name DESC",List the name and tonnage ordered by in descending alphaetical order for the names.,spider
2,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}","SELECT name , date FROM battle","List the name, date and result of each battle.",spider


In [11]:
assert(merged_samples.isnull().sum().sum() == 0)

def ensure_question(question):
    if len(question) > 0 and question[-1] != '?':
        question = question + "?"
    return question

# ensure questions end with question mark
merged_samples["question"] = merged_samples["question"].map(ensure_question)

In [33]:
from collections import defaultdict
from pathlib import Path
from datasets import Dataset, DatasetDict
import random
import numpy as np

SAMPLE_SIZE = 5 # Full DB takes ~26 hours on T4x2, reduce based on available capacity

db_ids = merged_samples["db_id"].unique()
db_ids = rng.permutation(db_ids)
db_ids = db_ids[:SAMPLE_SIZE]

train_ratio: float = 0.6
val_ratio: float = 0.3
seed: int = 42

n = len(db_ids)
print(f"Total DBs: {n}")
rng = np.random.default_rng(seed=seed)

print(db_ids)

n_train = max(1, int(n * train_ratio))
n_val = max(1, int(n * val_ratio))
train_dbs = db_ids[:n_train]
val_dbs = db_ids[n_train: n_train + n_val]
test_dbs = db_ids[n_train + n_val:]

print(f"DBs in each set: Train, Val, Test: {len(train_dbs), len(val_dbs), len(test_dbs)}")
train_ds = merged_samples[merged_samples["db_id"].isin(train_dbs)]
val_ds = merged_samples[merged_samples["db_id"].isin(val_dbs)]
test_ds = merged_samples[merged_samples["db_id"].isin(test_dbs)]

print(f"Rows in each set: Train, Val, Test: {len(train_ds), len(val_ds), len(test_ds)}")

Total DBs: 5
['wta_1' 'network_1' 'cre_Doc_Template_Mgt' 'toxicology'
 'thrombosis_prediction']
DBs in each set: Train, Val, Test: (3, 1, 1)
Rows in each set: Train, Val, Test: (202, 147, 166)


In [34]:
merged_samples.head()

,db_id,schema,SQL,question,source
0,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}",SELECT count(*) FROM ship WHERE disposition_of_ship = 'Captured',How many ships ended up being 'Captured'?,spider
1,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}","SELECT name , tonnage FROM ship ORDER BY name DESC",List the name and tonnage ordered by in descending alphaetical order for the names.?,spider
2,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}","SELECT name , date FROM battle","List the name, date and result of each battle.?",spider
3,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}","SELECT max(killed) , min(killed) FROM death",What is maximum and minimum death toll caused each time?,spider
4,battle_death,"{'battle': ['id', 'name', 'date', 'bulgarian_commander', 'latin_commander', 'result'], 'ship': ['lost_in_battle', 'id', 'name', 'tonnage', 'ship_type', 'location', 'disposition_of_ship'], 'death': ['caused_by_ship_id', 'id', 'note', 'killed', 'injured']}",SELECT avg(injured) FROM death,What is the average number of injuries caused each time?,spider


---

## 🎯 Section 5 — Reward Functions

Define the three reward components used to guide GRPO training:
- **`format_reward`** — checks for valid SQL code block + parseability
- **`exec_reward`** — executes SQL against SQLite and checks for errors
- **`schema_fidelity_reward`** — measures fraction of valid table/column references

In [36]:
from __future__ import annotations
import re
import sqlite3
import sys
from typing import Any
import sqlglot
from loguru import logger


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

_SQL_FENCE_RE  = re.compile(r"```(?:sql)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)
_INLINE_SQL_RE = re.compile(r"(SELECT\s+.+?;)",        re.DOTALL | re.IGNORECASE)

SUPPORTED_DIALECTS = ("sqlite", "duckdb", "postgres", "mysql", "tsql", "bigquery")


def extract_sql(text: str) -> str | None:
    """Return the first SQL block found in *text*, or ``None``."""
    m = _SQL_FENCE_RE.search(text)
    if m:
        return m.group(1).strip()
    m = _INLINE_SQL_RE.search(text)
    if m:
        return m.group(1).strip()
    return None


# ---------------------------------------------------------------------------
# format_reward
# ---------------------------------------------------------------------------

def format_reward(
    completions: list[list[dict[str, str]]],
    **kwargs: Any,
) -> list[float]:
    """Return 1.0 for each completion that contains parseable SQL, else 0.0."""
    rewards: list[float] = []
    for idx, messages in enumerate(completions):
        text = messages[-1]["content"] if messages else ""
        sql  = extract_sql(text)
        if sql is None:
            logger.warning(f"  [{idx}] No SQL block found")
            rewards.append(0.0)
            continue
        try:
            parsed = sqlglot.parse(sql, error_level=sqlglot.ErrorLevel.RAISE)
            score  = 1.0 if parsed else 0.0
        except sqlglot.errors.ParseError as exc:
            score = 0.0
        rewards.append(score)
    return rewards


# ---------------------------------------------------------------------------
# exec_reward
# ---------------------------------------------------------------------------
# /kaggle/working/rawdata/bird/dev_20240627/

def _exec_on_sqlite(sql: str, db_path: str | None = None, source: str | None = None, base_path = "/kaggle/working/") -> tuple[bool, str | None]:
    """Try to execute *sql*. Returns (success, error_message)."""
    conn = None
    if source == "spider" and db_path:
        db_path = base_path + "rawdata/spider/spider_data/database/" + db_path + f"/{db_path}.sqlite"
        conn = sqlite3.connect(db_path or ":memory:")
        try:
            conn.execute(sql)
            return True, None
        except Exception as exc:  # noqa: BLE001
            return False, str(exc)
        finally:
            conn.close()
    elif source =="bird" and db_path:
        db_path = base_path + "rawdata/bird/dev_databases/" + db_path + f"/{db_path}.sqlite"
        conn = sqlite3.connect(db_path or ":memory:")
        try:
            conn.execute(sql)
            return True, None
        except Exception as exc:  # noqa: BLE001
            return False, str(exc)
        finally:
            conn.close()
    else:
        logger.error(f"Failed to validate execution for {db_path}, {source}")
        return False, "Not executed"

def exec_reward(
    completions: list[list[dict[str, str]]],
    prompts:     list[list[dict[str, str]]] | None = None,
    dialect:     str                               = "sqlite",
    db_paths:    list[str | None]          | None  = None,
    source:    list[str | None]          | None  = None,
    **kwargs: Any,
) -> list[float]:
    """Execute each SQL and return -1.0 (no SQL) / 0.0 (error) / 1.0 (success)."""
    if dialect not in SUPPORTED_DIALECTS:
        raise ValueError(f"Unsupported dialect '{dialect}'. Choose from {SUPPORTED_DIALECTS}.")

    rewards: list[float] = []
    paths   = db_paths if db_paths is not None else [None] * len(completions)

    for idx, messages in enumerate(completions):
        text = messages[-1]["content"] if messages else ""
        sql  = extract_sql(text)

        if sql is None:
            logger.warning(f"  [{idx}] No SQL found → score=-1.0")
            rewards.append(-1.0)
            continue

        if dialect != "sqlite":
            try:
                sql = sqlglot.transpile(sql, read=dialect, write="sqlite")[0]
            except sqlglot.errors.ParseError as exc:
                logger.warning(f"  [{idx}] Transpile error: {exc} → score=0.0")
                rewards.append(0.0)
                continue

        ok, err = _exec_on_sqlite(sql, paths[idx], source[idx])
        score   = 1.0 if ok else 0.0
        if ok:
            pass
        else:
            logger.warning(f"  [{idx}] Execution failed: {err} → score={score}")
        rewards.append(score)
    return rewards

# ---------------------------------------------------------------------------
# schema_fidelity_reward
# ---------------------------------------------------------------------------

def _extract_schema_items(sql: str) -> tuple[set[str], set[str]]:
    """Return (tables, columns) referenced in *sql* (lowercased)."""
    tables:  set[str] = set()
    columns: set[str] = set()
    try:
        for stmt in sqlglot.parse(sql):
            if stmt is None:
                continue
            for node in stmt.walk():
                if isinstance(node, sqlglot.exp.Table)  and node.name:
                    tables.add(node.name.lower())
                if isinstance(node, sqlglot.exp.Column) and node.name:
                    columns.add(node.name.lower())
    except sqlglot.errors.ParseError as exc:
        logger.warning(f"  schema parse error: {exc}")
    return tables, columns


def schema_fidelity_reward(
    completions: list[list[dict[str, str]]],
    schemas:     list[dict[str, list[str]]] | None = None,
    **kwargs: Any,
) -> list[float]:
    """Reward = fraction of referenced tables/columns that exist in the schema."""
    n           = len(completions)
    schema_list = schemas if schemas is not None else [{}] * n
    rewards: list[float] = []

    for idx, messages in enumerate(completions):
        text   = messages[-1]["content"] if messages else ""
        sql    = extract_sql(text)
        schema = schema_list[idx] if idx < len(schema_list) else {}

        if sql is None:
            rewards.append(0.0)
            continue

        if not schema:
            rewards.append(0.5)
            continue

        tables_in_schema  = set(schema.keys())
        columns_in_schema = {col for cols in schema.values() for col in cols}
        ref_tables, ref_columns = _extract_schema_items(sql)

        all_refs   = ref_tables | ref_columns
        valid_refs = (ref_tables & tables_in_schema) | (ref_columns & columns_in_schema)
        score      = len(valid_refs) / len(all_refs) if all_refs else 0.5
        rewards.append(score)
    return rewards


# ---------------------------------------------------------------------------
# combined_reward
# ---------------------------------------------------------------------------

def combined_reward(
    completions: list[list[dict[str, str]]],
    prompts:     list[list[dict[str, str]]] | None  = None,
    schemas:     list[dict[str, list[str]]] | None  = None,
    dialect:     str                                = "sqlite",
    db_paths:    list[str | None]           | None  = None,
    source:      list[str | None]           | None  = None,
    weights:     dict[str, float]           | None  = None,
    **kwargs: Any,
) -> list[float]:
    """Weighted combination of format + exec + schema_fidelity rewards.

    Default weights: format=0.2, exec=0.5, schema_fidelity=0.3.

    Parameters
    ----------
    log:
        Override the global ``LOG_COMBINED_REWARD`` flag at call-site.
    """
    w = weights or {"format": 0.2, "exec": 0.5, "schema_fidelity": 0.3}
    fmt = format_reward(completions)
    exc = exec_reward(completions, prompts=prompts, dialect=dialect, db_paths=db_paths, source=source)
    sfr = schema_fidelity_reward(completions, schemas=schemas)

    results: list[float] = []
    for idx, (f, e, s) in enumerate(zip(fmt, exc, sfr)):
        combined = round(w["format"] * f + w["exec"] * e + w["schema_fidelity"] * s, 4)
        results.append(combined)
    return results


# ---------------------------------------------------------------------------
# Test Spider
# ---------------------------------------------------------------------------

ACADEMIC_SCHEMA = {
    "author":             ["aid", "homepage", "name", "oid"],
    "cite":               ["cited", "citing"],
    "data":               ["did", "journal_id", "keyword_id"],
    "domain":             ["did", "name"],
    "domain_author":      ["aid", "did"],
    "domain_conf":        ["cid", "did"],
    "domain_journal":     ["did", "jid"],
    "domain_keyword":     ["did", "kid"],
    "domain_publication": ["did", "pid"],
    "field":              ["fid", "field"],
    "journal":            ["homepage", "jid", "name"],
    "keyword":            ["keyword", "kid"],
    "organization":       ["continent", "homepage", "name", "oid"],
    "publication":        ["abstract", "cid", "citation_num", "jid", "pid",
                           "reference_num", "title", "year"],
    "publication_keyword":["kid", "pid"],
    "writes":             ["aid", "pid"],
    "conference":         ["cid", "homepage", "name"],
}

test_completions = [[{
    "role": "assistant",
    "content": (
        "```sql\nSELECT a.name, COUNT(w.pid) AS pub_count FROM author a JOIN writes w ON a.aid = w.aid GROUP BY a.aid, a.name HAVING COUNT(w.pid) > 5 ORDER BY pub_count DESC;\n```"
    ),
}]]

test_prompts = [[
    {
        "role": "system",
        "content": "You are an expert SQL assistant. Return ONLY the SQL inside ```sql``` blocks.",
    },
    {
        "role": "user",
        "content": (
            "### Question\n"
            "Which authors have written more than 5 publications?\n"
            f"### Schema\n{ACADEMIC_SCHEMA}"
        ),
    },
]]

test_schemas = [ACADEMIC_SCHEMA]
test_source = ["spider"]
test_db_ids = ["academic"]
scores = combined_reward(test_completions, test_prompts, test_schemas, db_paths=test_db_ids, source=test_source, log=True)
print(f"\nFinal scores for Spider db: {scores}")

# ---------------------------------------------------------------------------
# Test Bird
# Name of the table Transactions is Transactions_1k 
# ---------------------------------------------------------------------------

DEBIT_CARD_SCHEMA = {
    "customers":       ["CustomerID", "Segment", "Currency"],
    "gasstations":     ["GasStationID", "ChainID", "Country", "Segment"],
    "products":        ["ProductID", "Description"],
    "transactions_1k":    ["TransactionID", "Date", "Time", "CustomerID", "CardID", 
                        "GasStationID", "ProductID", "Amount", "Price"],
    "yearmonth":       ["CustomerID", "Date", "Consumption"],
}

test_completions = [[{
    "role": "assistant",
    "content": (
        "```sql\nSELECT c.Segment, SUM(t.Amount) AS total_spent FROM customers c JOIN transactions_1k t ON c.CustomerID = t.CustomerID GROUP BY c.Segment ORDER BY total_spent DESC;\n```"
    ),
}]]

test_prompts = [[
    {
        "role": "system",
        "content": "You are an expert SQL assistant. Return ONLY the SQL inside ```sql``` blocks.",
    },
    {
        "role": "user",
        "content": (
            "### Question\n"
            "What is the total amount spent by each customer segment?\n"
            f"### Schema\n{DEBIT_CARD_SCHEMA}"
        ),
    },
]]

test_schemas = [DEBIT_CARD_SCHEMA]
test_source = ["bird"]
test_db_ids = ["debit_card_specializing"]
scores = combined_reward(test_completions, test_prompts, test_schemas, db_paths=test_db_ids, source=test_source, log=True)
print(f"\nFinal scores for bird db: {scores}")


Final scores for Spider db: [0.95]

Final scores for bird db: [0.8]


### TBD - Create Baseline using LLM here ---

---

## 🏋️ Section 6 — GRPO Fine-tuning

Fine-tune the SLM using Group Relative Policy Optimization (GRPO) with the combined reward signal.  
The model learns to prefer SQL outputs that are well-formatted, executable, and schema-faithful.

In [37]:
from unsloth import FastLanguageModel  
from trl import GRPOConfig
import yaml

grpo_config = """
# GRPO algorithm configuration
# All values here mirror TRL GRPOConfig fields

model:
  name_or_path: "unsloth/Qwen2.5-3B-Instruct"
  load_in_4bit: true
  use_gradient_checkpointing: "unsloth"

grpo:
  # Number of rollout samples per prompt
  num_generations: 3
  # Max tokens to generate during rollout
  max_new_tokens: 512
  # Temperature for sampling
  temperature: 0.7
  # KL penalty coefficient
  beta: 0.04
  # Clip range for the policy ratio
  epsilon: 0.2
  # Number of PPO-style update steps per batch
  num_iterations: 1
  # Use DAPO loss (clip-higher variant)
  use_dapo: false

tokenizer:
  max_length: 2048
  padding_side: "left"

reward:
  # Weights file location (overridden at runtime)
  weights_file: "configs/reward_weights.yaml"
"""


training_args = """
# HuggingFace TrainingArguments / TRL GRPOTrainer settings
output_dir: "/kaggle/working/grpo_checkpoints"
run_name: "text2sql-grpo"

# Batch sizes
per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8

# Learning rate schedule
learning_rate: 2.0e-5
lr_scheduler_type: "cosine"
warmup_ratio: 0.05

# Training length
num_train_epochs: 1
max_steps: -1

# Precision
bf16: false
fp16: false

# Checkpointing
save_strategy: "steps"
save_steps: 100
save_total_limit: 3
evaluation_strategy: "steps"
eval_steps: 100
load_best_model_at_end: true

# Logging
logging_steps: 10

# Optimizer
optim: "adamw_8bit"
weight_decay: 0.01
max_grad_norm: 1.0

# Data
dataloader_num_workers: 4
remove_unused_columns: false

# Misc
seed: 42
data_seed: 42
"""

reward_weights = """
# Reward function weights – must sum to 1.0
#
# format_reward        : SQL syntax validity (sqlglot parse succeeds)
# exec_reward          : Query executes without error on target DB
# schema_fidelity_reward: All referenced tables/columns exist in the schema

format_reward: 0.2
exec_reward: 0.5
schema_fidelity_reward: 0.3

# Penalty applied when model output contains no SQL block at all
no_sql_penalty: -1.0

# Penalty for SQL that parses but references unknown schema items
unknown_schema_item_penalty: -0.5
"""

def _load_yaml(config: str) -> dict[str, Any]:
    return yaml.safe_load(config)
    
grpo_cfg = _load_yaml(grpo_config)
train_cfg = _load_yaml(training_args)
reward_cfg = _load_yaml(reward_weights)
reward_weights = {
        "format": reward_cfg.get("format_reward", 0.2),
        "exec": reward_cfg.get("exec_reward", 0.5),
        "schema_fidelity": reward_cfg.get("schema_fidelity_reward", 0.3),
}

In [38]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = grpo_cfg["model"]["name_or_path"],
    max_seq_length = grpo_cfg["tokenizer"]["max_length"],
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", # Uncomment based on processing power
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

INFO 03-09 12:57:57 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
INFO 03-09 12:57:57 [vllm_utils.py:753] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. However your setting of `gpu_memory_utilization` will OOM.
Changing `gpu_memory_utilization` to 0.8.
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 43.46%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 20

RuntimeError: Sleep mode can only be used for one instance per process.

In [39]:
_SYSTEM_PROMPT = (
    "You are an expert SQL assistant. Given a database schema and a natural language "
    "question, write a correct SQL query that answers the question.\n"
    "Return ONLY the SQL query inside a ```sql ... ``` code block."
)

def build_prompt(
    question: str,
    schema: str,
    system_prompt: str = _SYSTEM_PROMPT,
) -> str:
    parts = []
    parts.append(f"### Question\n{question}")
    parts.append(f"### Schema\n{schema}")
    prompt = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": "\n".join(parts)},
    ]
    return prompt


def _make_prompt(question, schema, answer, source, db_id):
    prompt = build_prompt(question, schema)
    return {"prompt": prompt, "solution": answer, "schema": schema, "source": source, "db_id": db_id }

train_dataset = train_ds.apply(lambda row: _make_prompt(row["question"], row["schema"], row["SQL"], row["source"], row["db_id"]), axis=1).to_list() 
val_dataset = val_ds.apply(lambda row: _make_prompt(row["question"], row["schema"], row["SQL"], row["source"], row["db_id"]), axis=1).to_list() 
test_dataset = test_ds.apply(lambda row: _make_prompt(row["question"], row["schema"], row["SQL"], row["source"], row["db_id"]), axis=1).to_list() 

In [48]:
print(f"Sample one row from dataset")
train_dataset[0]

Sample one row from dataset


{'prompt': [{'role': 'system',
   'content': 'You are an expert SQL assistant. Given a database schema and a natural language question, write a correct SQL query that answers the question.\nReturn ONLY the SQL query inside a ```sql ... ``` code block.'},
  {'role': 'user',
   'content': "### Question\nHow many documents do we have?\n### Schema\n{'ref_template_types': ['template_type_code', 'template_type_description'], 'templates': ['template_id', 'version_number', 'template_type_code', 'date_effective_from', 'date_effective_to', 'template_details'], 'documents': ['document_id', 'template_id', 'document_name', 'document_description', 'other_details'], 'paragraphs': ['paragraph_id', 'document_id', 'paragraph_text', 'other_details']}"}],
 'solution': 'SELECT count(*) FROM Documents',
 'schema': {'ref_template_types': ['template_type_code',
   'template_type_description'],
  'templates': ['template_id',
   'version_number',
   'template_type_code',
   'date_effective_from',
   'date_effec

In [41]:
from trl import GRPOTrainer 

# ── Reward wrapper ─────────────────────────────────────
def reward_fn(
        completions: list[list[dict[str, str]]],
        prompts: list[list[dict[str, str]]] | None = None,
        **kw: Any,
    ) -> list[float]:
   schemas = kw["schema"]
   source = kw["source"] # schema
   db_ids = kw["db_id"] # table names
   return combined_reward(completions, prompts=prompts, schemas=schemas, source=source, db_paths=db_ids, weights=reward_weights)

grpo_train_cfg = GRPOConfig(
            output_dir=output_dir,
            num_generations=grpo_cfg["grpo"]["num_generations"],
            temperature=grpo_cfg["grpo"]["temperature"],
            beta=grpo_cfg["grpo"]["beta"],
            epsilon=grpo_cfg["grpo"]["epsilon"],
            num_iterations=1,
            per_device_train_batch_size=train_cfg.get("per_device_train_batch_size", 3),
            gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 8),
            learning_rate=train_cfg.get("learning_rate", 2e-5),
            num_train_epochs=train_cfg.get("num_train_epochs", 1),
            bf16=train_cfg.get("bf16", True),
            logging_steps=1,
            report_to = "none",
            save_steps=train_cfg.get("save_steps", 15)
)

trainer = GRPOTrainer(
            model=model,
            tokenizer=tokenizer,
            reward_funcs=[reward_fn],
            args=grpo_train_cfg,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
)

trainer_stats = trainer.train()

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 3


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 202 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 3 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (3 x 8 x 1) = 24
 "-____-"     Trainable parameters = 119,734,272 of 3,205,672,960 (3.74% trained)
2026-03-09 12:58:57.342 | WARNING  | __main__:exec_reward:126 -   [4] Execution failed: no such column: m.winner_rank_points → score=0.0
2026-03-09 12:58:57.345 | WARNING  | __main__:exec_reward:126 -   [6] Execution failed: no such function: YEAR → score=0.0
2026-03-09 12:58:57.346 | WARNING  | __main__:exec_reward:126 -   [7] Execution failed: no such function: YEAR → score=0.0
2026-03-09 12:58:57.347 | WARNING  | __main__:exec_reward:126 -   [8] Execution failed: no such function: YEAR → score=0.0


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
1,0.002300,0.893529,0.038011,49.958336,20.000000,99.000000,0.000000,49.958336,20.000000,99.000000,0.000004,0.893529,0.215616
2,0.005200,0.979167,0.036084,47.791668,25.000000,80.000000,0.000000,47.791668,25.000000,80.000000,0.000011,0.979167,0.102062
3,0.041300,0.955833,0.076499,42.750000,27.000000,85.000000,0.000000,42.750000,27.000000,85.000000,0.000055,0.955833,0.149896
4,0.019600,0.991146,0.003608,62.416668,29.000000,256.000000,0.041667,54.000000,29.000000,113.000000,0.000659,0.991146,0.017861
5,0.065300,0.740596,0.119250,45.166668,23.000000,95.000000,0.000000,45.166668,23.000000,95.000000,0.005203,0.740596,0.514092
6,-0.001100,0.979167,0.036084,36.208336,19.000000,75.000000,0.000000,36.208336,19.000000,75.000000,0.009090,0.979167,0.102062
7,-0.019400,0.916667,0.036084,50.208336,20.000000,97.000000,0.000000,50.208336,20.000000,97.000000,0.021604,0.916667,0.190347
8,0.045600,0.954171,0.072169,60.333336,14.000000,196.000000,0.000000,60.333336,14.000000,196.000000,0.016307,0.954171,0.145396
9,-0.040300,0.979167,0.036084,37.125000,18.000000,97.000000,0.000000,37.125000,18.000000,97.000000,0.054853,0.979167,0.102062
10,0.045000,0.840833,0.037528,63.208336,18.000000,256.000000,0.041667,54.826088,18.000000,168.000000,0.075276,0.840833,0.255069


2026-03-09 13:00:00.877 | WARNING  | __main__:exec_reward:126 -   [4] Execution failed: no such column: f.liked_id → score=0.0
2026-03-09 13:01:52.178 | WARNING  | __main__:exec_reward:126 -   [15] Execution failed: no such column: f.id → score=0.0
2026-03-09 13:01:52.180 | WARNING  | __main__:exec_reward:126 -   [18] Execution failed: ambiguous column name: document_id → score=0.0
2026-03-09 13:03:10.147 | WARNING  | __main__:format_reward:45 -   [0] No SQL block found
2026-03-09 13:03:10.148 | WARNING  | __main__:format_reward:45 -   [1] No SQL block found
2026-03-09 13:03:10.148 | WARNING  | __main__:format_reward:45 -   [2] No SQL block found
2026-03-09 13:03:10.163 | WARNING  | __main__:exec_reward:109 -   [0] No SQL found → score=-1.0
2026-03-09 13:03:10.164 | WARNING  | __main__:exec_reward:109 -   [1] No SQL found → score=-1.0
2026-03-09 13:03:10.165 | WARNING  | __main__:exec_reward:109 -   [2] No SQL found → score=-1.0
2026-03-09 13:03:10.167 | WARNING  | __main__:exec_reward

In [49]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "### Question\nHow many documents do we have?\n### Schema\n{'ref_template_types': ['template_type_code', 'template_type_description'], 'templates': ['template_id', 'version_number', 'template_type_code', 'date_effective_from', 'date_effective_to', 'template_details'], 'documents': ['document_id', 'template_id', 'document_name', 'document_description', 'other_details'], 'paragraphs': ['paragraph_id', 'document_id', 'paragraph_text', 'other_details']}"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Based on the schema provided, the information about the number of documents is not directly stored in the schema structure. The schema describes the structure of the database, but it doesn't directly contain a count or list of documents.\n\nTo answer this question, we would need to query the database and count the number of documents. Since the schema is provided and not a database query result, I can't give you the exact count, but I can explain that the count would be found in a 'documents' table or collection, which stores each document's ID, template ID, document name, and other details.\n\nIf you have access to the database, you would run a query like this to find the total number of documents:\n\n```sql\nSELECT COUNT(*) FROM documents;\n```\n\nIf you're using a different database system or a programming language, the query would look similar but might be expressed differently."

In [50]:
model.save_lora("grpo_saved_lora")

In [53]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : _SYSTEM_PROMPT},
    {"role" : "user", "content" : "### Question\nHow many documents do we have?\n### Schema\n{'ref_template_types': ['template_type_code', 'template_type_description'], \
    'templates': ['template_id', 'version_number', 'template_type_code', 'date_effective_from', 'date_effective_to', 'template_details'], 'documents': ['document_id', 'template_id', 'document_name', 'document_description', 'other_details'], 'paragraphs': ['paragraph_id', 'document_id', 'paragraph_text', 'other_details']}"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'```sql\nSELECT COUNT(DISTINCT document_id) AS document_count\nFROM documents;\n```'